# ZotVision — CNN-ViT Hybrid Training (TPU)
Runtime → Change runtime type → TPU before running.

In [ ]:
!pip install -q efficientnet_pytorch transformers
!git clone https://github.com/varshinivij/ZotVision.git
!git -C /content/ZotVision checkout hoon
import sys
sys.path.append('/content/ZotVision/zot-vision/backend')

: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile

DRIVE_ZIP_PATH = '/content/drive/MyDrive/datasets.zip'  # update if needed
EXTRACT_PATH   = '/content/datasets'

if not os.path.exists(EXTRACT_PATH):
    print('Extracting dataset...')
    with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_PATH)
    print('Done.')
else:
    print('Dataset already extracted.')

IMAGES_DIR  = os.path.join(EXTRACT_PATH, 'images')
LABELS_FILE = os.path.join(EXTRACT_PATH, 'results', 'labels.txt')
WEIGHTS_DIR = '/content/drive/MyDrive/zotvision_weights'
os.makedirs(WEIGHTS_DIR, exist_ok=True)
print(f'Images: {len(os.listdir(IMAGES_DIR))} files')
print(f'Weights will save to: {WEIGHTS_DIR}')

In [ ]:
import torch
import torch_xla
import torch_xla.core.xla_model as xm

DEVICE = xm.xla_device()
print(f'Device: {DEVICE}')

In [ ]:
# Override paths before importing so transformer.py uses our Colab paths
import transformer
transformer.IMAGES_DIR  = IMAGES_DIR
transformer.LABELS_FILE = LABELS_FILE
transformer.DEVICE      = DEVICE

from transformer import (
    CNNViTHybrid, CustomDataset, load_samples, get_transforms,
    LABEL_MAP, NUM_CLASSES, BATCH_SIZE, NUM_EPOCHS, LR
)

import random, torch.nn as nn
from torch.utils.data import DataLoader

all_samples = load_samples(LABELS_FILE, IMAGES_DIR)
random.shuffle(all_samples)
split = int(0.8 * len(all_samples))
train_ds = CustomDataset(all_samples[:split], get_transforms(True))
val_ds   = CustomDataset(all_samples[split:],  get_transforms(False))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

model = CNNViTHybrid().to(DEVICE)
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
WEIGHTS_PATH = os.path.join(WEIGHTS_DIR, 'model_weights.pth')
best_val_acc = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    tl, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        xm.optimizer_step(optimizer)
        xm.mark_step()
        tl      += loss.item() * imgs.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total   += imgs.size(0)
    train_loss, train_acc = tl/total, correct/total

    model.eval()
    tl, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs)
            tl      += criterion(logits, labels).item() * imgs.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total   += imgs.size(0)
            xm.mark_step()
    val_loss, val_acc = tl/total, correct/total
    scheduler.step()

    print(f'Epoch {epoch:03d}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f}')
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        xm.save(model.state_dict(), WEIGHTS_PATH)
        print(f'  Saved best model (val_acc={best_val_acc:.4f}) -> {WEIGHTS_PATH}')

print(f'Training complete. Best Val Acc: {best_val_acc:.4f}')